# 2. 설정과 데이터 읽어오기

In [1]:
!pip install pandas numpy matplotlib scikit-learn xgboost

## 1. 라이브러리 import

In [2]:
import pandas as pd # 표 형태 데이터를 다루는 라이브러리
import numpy as np # 수치 계산용 라이브러리 (RMSE 계산 등에 사용)
import matplotlib.pyplot as plt # 그래프 시각화 라이브러리

# train_test_split: 데이터를 학습용곽 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split
# 회귀 모델 평가에 사용할 지표들(MAE, MAE→RMSE, R2)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# ColumnTransformer: 컬럼 종류별로 다른 전처리를 적용하는 도구
from sklearn.compose import ColumnTransformer
# Pipeline: 전처리와 모델 학습을 하나의 흐름으로 묶는 도구
from sklearn.pipeline import Pipeline
# OneHotEncoder: 문자형(범주형) 값을 숫자 형태로 바꾸는 도구
from sklearn.preprocessing import OneHotEncoder
# SimpleImputer: 비어 있는 값(결측치)을 정해진 규칙으로 채우는 도구
from sklearn.impute import SimpleImputer

# XGBRegressor: 숫자 값을 예측하는 회귀용 XGBoost모델
from xgboost import XGBRegressor

In [3]:
import pandas as pd# 표 형태 데이터를 다루는 라이브러리
import numpy as np# 수치 계산용 라이브러리 (RMSE 계산 등에 사용)
import matplotlib.pyplot as plt

# train_test_split: 데이터를 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split
# 회귀 모델 평가에 사용할 지표들(MAE, MAE→RMSE, R2)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# ColumnTransformer: 컬럼 종류별로 다른 전처리를 적용하는 도구
from sklearn.compose import ColumnTransformer
# Pipeline: 전처리와 모델 학습을 하나의 흐름으로 묶는 도구
from sklearn.pipeline import Pipeline
# OneHotEncoder: 문자형(범주형) 값을 숫자 형태로 바꾸는 도구
from sklearn.preprocessing import OneHotEncoder
# SimpleImputer: 비어 있는 값(결측치)을 정해진 규칙으로 채우는 도구
from sklearn.impute import SimpleImputer

# XGBRegressor: 숫자 값을 예측하는 회귀용 XGBoost모델
from xgboost import XGBRegressor

In [4]:
# 그래프의 모양을 설정 합니다.
plt.style.use("ggplot")

In [5]:
import matplotlib.font_manager as fm
import warnings

# ---------------------------------------
# Windows 한글 폰트 설정
# ---------------------------------------
# matplotlib의 기본 글꼴은 Dejavu Sans인데, 이 글꼴을 한글을 제대로 표시하지 못합니다.
# 그래서 Windows에 기본으로 설치된 '맑은 고딕'을 그래프 글꼴로 저장합니다.
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스(-) 기호가 깨지는 문제를 방지합니다.
# 예: -10 같은 값이 □10 처럼 보이는 문제 예방
plt.rcParams['axes.unicode_minus'] = False

# 한글 글꼴 관련 경고가 너무 많이 출력되는 것을 줄입니다.
warnings.filterwarnings('ignore', category = UserWarning)

## 2. 데이터 로드

In [6]:
# 같은 폴더에 jeju_bus.csv 가 있다고 가정합니다.
# 파일이 다른 경로에 있다면 아래 경로를 수정합니다. 예: pd.read_csv('data/jeju_bus.csv')
df = pd.read_csv('jeju_bus.csv') # csv 파일을 읽어 DataFrame(df)으로 만듭니다.

# 원본 데이터는 보존하고, 모델링용으로는 복사본을 사용합니다.
# .copy()를 쓰면 df_model을 바꿔도 원본 df는 영향을 받지 않습니다.
df_model = df.copy()

## 3. 데이터 기본 확인

In [7]:
# 상위 5개 행을 확인합니다. 실제 값이 어떤 형태인지 눈으로 점검하는 단계입니다.
df.head()

,id,date,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,now_arrive_time,distance,next_station,next_latitude,next_longitude,next_arrive_time
0,0,2019-10-15,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,06시,266.0,제대마을,33.457724,126.554014,24
1,1,2019-10-15,405136001,7997025,360-1,33.457724,126.554014,제대마을,06시,333.0,제대아파트,33.458783,126.557353,36
2,2,2019-10-15,405136001,7997025,360-1,33.458783,126.557353,제대아파트,06시,415.0,제주대학교,33.459893,126.561624,40
3,3,2019-10-15,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),06시,578.0,제주여자중고등학교(아라방면),33.484860,126.542928,42
4,4,2019-10-15,405136001,7997025,360-1,33.485662,126.494923,도호동,07시,374.0,은남동,33.485822,126.490897,64


In [8]:
# 데이터의 (행 개수, 열 개수)를 확인합니다. 데이터 규모를 파악합니다.
df.shape

(210457, 14)

In [9]:
# 실제 컬럼명을 확인합니다. 이후 코드(전처리, feature 목록 등)는 이 컬럼명을 기준으로 작성되어 있습니다.
# 만약 실제 컬럼명이 코드와 다르면, 이 출력을 기준으로 코드의 컬럼명을 맞춰야 합니다.
df.columns

Index(['id', 'date', 'route_id', 'vh_id', 'route_nm', 'now_latitude',
       'now_longitude', 'now_station', 'now_arrive_time', 'distance',
       'next_station', 'next_latitude', 'next_longitude', 'next_arrive_time'],
      dtype='str')

In [10]:
# 컬럼별 결측치(비어 있는 값) 개수를 확인합니다. 값이 클수록 그 컬럼에 빈 값이 많다는 뜻입니다.
df.isnull().sum()

id                  0
date                0
route_id            0
vh_id               0
route_nm            0
now_latitude        0
now_longitude       0
now_station         0
now_arrive_time     0
distance            0
next_station        0
next_latitude       0
next_longitude      0
next_arrive_time    0
dtype: int64

# 4. 기본 정보 확인

## df.info()란 무엇인가요?

In [11]:
# 각 컬럼의 자료형(숫자/문자)과 결측치 여부를 한눈에 확인합니다.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 210457 entries, 0 to 210456
Data columns (total 14 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                210457 non-null  int64  
 1   date              210457 non-null  str    
 2   route_id          210457 non-null  int64  
 3   vh_id             210457 non-null  int64  
 4   route_nm          210457 non-null  str    
 5   now_latitude      210457 non-null  float64
 6   now_longitude     210457 non-null  float64
 7   now_station       210457 non-null  str    
 8   now_arrive_time   210457 non-null  str    
 9   distance          210457 non-null  float64
 10  next_station      210457 non-null  str    
 11  next_latitude     210457 non-null  float64
 12  next_longitude    210457 non-null  float64
 13  next_arrive_time  210457 non-null  int64  
dtypes: float64(5), int64(4), str(5)
memory usage: 22.5 MB


# 5. 데이터 확인하기

In [12]:
# 수치형 컬럼의 요약 통계(평균, 표준편차, 최솟값/최댓값, 분위수)를 확인합니다.
df.describe()

,id,route_id,vh_id,now_latitude,now_longitude,distance,next_latitude,next_longitude,next_arrive_time
count,210457.000000,2.104570e+05,2.104570e+05,210457.000000,210457.000000,210457.000000,210457.000000,210457.000000,210457.000000
mean,105228.000000,4.052491e+08,7.988694e+06,33.434528,126.603451,490.256100,33.434711,126.603687,85.380824
std,60753.847139,9.132404e+04,6.774077e+03,0.102350,0.123961,520.563932,0.102224,0.123838,85.051170
min,0.000000,4.051360e+08,7.983000e+06,33.244382,126.473300,97.000000,33.244382,126.473300,6.000000
25%,52614.000000,4.051365e+08,7.983093e+06,33.325283,126.523900,291.000000,33.325283,126.524550,44.000000
50%,105228.000000,4.053201e+08,7.983431e+06,33.484667,126.551050,384.000000,33.484860,126.551050,66.000000
75%,157842.000000,4.053201e+08,7.997041e+06,33.500197,126.650322,542.000000,33.500228,126.650322,102.000000
max,210456.000000,4.053281e+08,7.997124e+06,33.556167,126.935188,7461.000000,33.556167,126.935188,2996.000000


# 전체 컬럼명 확인
df.columns

# 6. 전처리

In [13]:
# 예측 대상(정답) 컬럼명을 변수로 지정해 둡니다. 이후 코드에서 이 변수를 일관되게 상용합니다.
target_col = df.columns[-1]
target_col

# 전처리 입력에 반드시 있어야 하는 필수 컬럼 목록입니다.
# 이 목록은 아래 prepare_features() 안에서 "누락된 컬럼이 없는지" 검사하는 데 사용됩니다.
# 실제 데이터 컬럼명이 다르면 df.columns 출력을 확인한 뒤 이 목록을 수정합니다.
required_cols = df.columns[1:-1]
required_cols

Index(['date', 'route_id', 'vh_id', 'route_nm', 'now_latitude',
       'now_longitude', 'now_station', 'now_arrive_time', 'distance',
       'next_station', 'next_latitude', 'next_longitude'],
      dtype='str')

In [14]:
def prepare_features(df_input):
    # 원본을 직접 수정하지 않도록 복사본을 만들어 작업합니다.
    # (함수 밖의 데이터가 의도치 않게 바뀌는 것을 막기 위함입니다.)
    data = df_input.copy()
    
    # 필수 컬럼이 바지지 않았는지 먼저 검사합니다.
    # 누락된 컬럼이 있으면 어떤 컬럼이 없는지 알려주는 오류를 발생시켜 원인을 빨리 찾게 합니다.
    missing_cols = [col for col in required_cols if col not in data.columns]
    if missing_cols:
        raise ValueError('missing cols : True!!')

    # date를 datetime(날짜형)으로 변환한 뒤, 모델이 활용하기 좋은 숫자 파생 컬럼은 만듭니다.
    data['date'] = pd.to_datetime(data['date'])
    data['year'] = data['date'].dt.year# 연도
    data['month'] = data['date'].dt.month# 월
    data['day'] = data['date'].dt.day# 연도
    data['dayofweek'] = data['date'].dt.dayofweek# 요일(월 = 0 ... 일 = 6)

    # now_arrive_time의 "06시" 같은 문자열에서 숫자 부분만 추출해 now_hour(정수 시간)로 만듭니다.
    # str.extract(r"(\d+)")는 문자열에서 연속된 숫자를 뽑아내는 처리입니다. ("06시"→"06")
    data['now_hour'] = (
        data['now_arrive_time'].str.extract(r'(\d+)').astype(float)# str: 데이터 1줄씩 문자열을 뽑아서 반복할 거야
    )

    # 변환을 참이 원본 date, noew_arrive_time 컬럼은 더 이상 필요 없으므로 feature에서 제외합니다.
    data = data.drop(columns = ['date', 'now_arrive_time'])

    # 정답(target) 컬럼이 들어 있으면 제거합니다. (입력 feature에는 정답이 포함되면 안 됩니다.)
    # 예측용 데이터에는 보통 target이 없으므로, 있을 때만 제거하도록 조건을 둡니다.
    if target_col in data.columns:
        data = data.drop(columns = [target_col])

    # 단순 식별 번호인 id 컬럼이 있으면 제거합니다. (예측에 도움이 되지 않는 정보입니다.)
    if "id" in data.columns:
        data = data.drop(columns = ["id"])

    return data

data = df.copy()

data.head()

data['date']

data['date'] = pd.to_datetime( data['date'] )

# 필수 컬럼에서 해당하는 컬럼을 제외하고 출력합니다.
[col for col in required_cols if col not in ['date', 'vh_id'] ]

data['date'].info()

data['date'].dt.year

data['date'].dt.month

data['date'].dt.day

data['date'].dt.dayofweek

data

data['now_arrive_time']

data

data['now_hour']

data.columns

for col in required_cols :
    print(f'col = {col}')
    print('=' * 30)

required_cols[ :-2]

missing_cols = [col for col in required_cols if col not in required_cols[ :-2]]
missing_cols

if missing_cols :
    print('missing cols는 True!!')
else:
    print('missing cols는 False')

if missing_cols:
    raise ValueError(f'필수 컬럼 누락!! : {missing_cols}')

data.columns

target_col

target_col in data.columns    

# 범주형 데이터 숫자 처리
snack_df = pd.DataFrame({
                '과자명': ['초코파이', '오예스', '꼬북칩', '짱구']
                })

snack_df

# OneHot Encoding 할 거다.
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown = 'ignore'))
])

# 과자명 컬럼 One Hot Encoding 할 거다.
preprocess = ColumnTransformer(transformers = [
    ('cat', categorical_transformer, ['과자명'])    
])

preprocess.fit_transform(snack_df)

pd.DataFrame(
    preprocess.fit_transform(snack_df).toarray(),
    columns = preprocess.get_feature_names_out()
)

In [15]:
# 모델링용 복사본을 사용합니다. (원본 df는 그대로 보존)
df_model = df.copy()

# 학습 데이터에도 동일한 prepare_features 함수를 적용해 입력 features(X)를 만듭니다.
X_preprocess = prepare_features(df_model)

In [16]:
X_preprocess.head(), X_preprocess.columns

(    route_id    vh_id route_nm  now_latitude  now_longitude now_station  \
 0  405136001  7997025    360-1     33.456267     126.551750     제주대학교입구   
 1  405136001  7997025    360-1     33.457724     126.554014        제대마을   
 2  405136001  7997025    360-1     33.458783     126.557353       제대아파트   
 3  405136001  7997025    360-1     33.479705     126.543811   남국원(아라방면)   
 4  405136001  7997025    360-1     33.485662     126.494923         도호동   
 
    distance     next_station  next_latitude  next_longitude  year  month  day  \
 0     266.0             제대마을      33.457724      126.554014  2019     10   15   
 1     333.0            제대아파트      33.458783      126.557353  2019     10   15   
 2     415.0            제주대학교      33.459893      126.561624  2019     10   15   
 3     578.0  제주여자중고등학교(아라방면)      33.484860      126.542928  2019     10   15   
 4     374.0              은남동      33.485822      126.490897  2019     10   15   
 
    dayofweek  now_hour  
 0          1       6.

In [17]:
# 전처리 후 모델에 들어갈 범주형 컬럼 목롭입니다. (이름/종류를 나타내는 문자형 컬럼들)
categorical_features = [
    'route_nm',# 버스노선 이름
    'now_station',# 현재 정류장
    'next_station'# 다음 정류장
]

# 범주형 전처리: 결측치를 최빈값(most_frequent)으로 채운 뒤 OneHotEncoder로 숫자화합니다.
# handle_unknown="ignore": 예측 시 학습 때 보지 못한 새 값이 와도 오류 없이 처리합니다.
# 버전 호환성을 위해 sparse_output / sparse 인자는 지정하지 않고 handle_unknown 만 사용합니다.
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# ColumnTransformer: 수치형 컬럼과 범주형 컬럼에 위에서 정의한 서로 다른 전처리를 한 번에 적용합니다.
# ("cat", 범주형 전처리, 범주형 컬럼들) 형태로 묶습니다.
preprocessor = ColumnTransformer(transformers=[
    ('cat', categorical_transformer, categorical_features)
])

In [18]:
# fir_transform()
# - fit: X안의 범주형 값 목록을 학습합니다.
#   예: toute_nm에는 어떤 노선명이 있는지, now_station에는 어떤 정류장이 있는지 확인
#
# - transform: 학습한 범주형 값 목록을 기준으로 OneHotEncoder 숫자 배열로 변환합니다.
X_cat_encoded = preprocessor.fit_transform(X_preprocess)

In [19]:
# preprocessor는 ColuymnTrnasformer 객체입니다.
# 위에서 preprocessor.fit_transform(X)를 실행하면,
# preprocessor는 X 안에 있는 범주형 컬럼드르이 실제 값들을 학습합니다.
#
# 예를 들어 route_nm 컬럼에 다음 값들이 있었다고 가정합니다.
# - 201번
# - 202번
# - 300번
#
# OneHotEncoder는 이 값들을 각각 별도의 컬럼으로 바꿉니다.
# 예:
# - cat__route_nm_201번
# - cat__route_nm_202번
# - cat__route_nm_300번
#
# get_feature_names_out()은 이렇게 새로 만들어진 컬럼 이름 목록을 가져오는 함수입니다.
encoded_feature_names = preprocessor.get_feature_names_out()

In [20]:
# X_cat_encoded는 preprocessor.fit_transform(X)의 결과입니다.
# 즉, route_nm, now_station, next_station 같은 문자형 컬럼들이
# OneHotEncoder를 거쳐 0과 1로 변환된 결과입니다.
#
# 다만 OneHotEncoder의 결과는 일반적인 표(DataFrane)가 아니라
# sparse matrix 형태일 수 있습니다.
#
# sparse matrix는 대부분의 값이 0일 때 메모리를 아끼기 위해 사용하는 저장 방식입니다.
# 예를 들어 정류장 종류가 1,000개라면,
# 한 행에서 실제로 1이 되는 컬럼은 일부이고 대부분은 0입니다.
#
# 이런 데이터를 모두 표 형태로 저장하면 메모리를 많이 사용하므로,
# scikit-learn은 기본적으로 sparse matrix 형태로 저장할 수 있습니다.
#
# 하짐나 수정 중에는 사람이 직접 눈으로 확인해야 하므로,
# toarray()를 사용해서 일반 numpy 배열 형태로 바꾼 뒤
# pandar DataFrame으로 변환합니다.
#
# columns=encoded_feature_names를 지정하면
# 0과 1로 바뀐 각 컬럼에 사람이 이해할 수 있는 이름이 붙습니다.
X_cat_encoded_df = pd.DataFrame(
    X_cat_encoded.toarray(),           # sparse matrix를 일반 배열로 변환
    columns = encoded_feature_names    # OneHotEncoder가 만든 컬럼 이름을 DataFrame 컬럼 명으로 사용    
)

In [21]:
# X_cat_encoded_df는 모델에 들어가기 직전의 범주형 전처리 결과입니다.
# 원래 문자였던 route_nm, now_station, nest_sation이
# 여러 개의 0/1 컬럼으로 바뀐 것을 확인할 수 있습니다.
#
# 값이 1이면 해당 행이 그 범주에 해당한다는 뜻입니다.
# 값이 0이면 해당 행이 그 범주에 해당하지 않는다는 뜻입니다.
#
# 예:
# cat__route_nm_201번 값이 1이면 → 해당 데이터의 노선명이 201번
# cat__now_station_제주공항 값이 1이면 → 현재 정류장이 제주공항
# cat__next_station_중앙로 값이 1이면 → 다음 정류장이 중앙로
X_cat_encoded_df

,cat__route_nm_201-11,cat__route_nm_201-12,cat__route_nm_201-13,cat__route_nm_201-14,cat__route_nm_201-15,cat__route_nm_201-16,cat__route_nm_201-17,cat__route_nm_201-18,cat__route_nm_201-21,cat__route_nm_201-22,...,cat__next_station_화북주공아파트,cat__next_station_화북초등학교,cat__next_station_화성농장,cat__next_station_효돈농협하나로마트,cat__next_station_효돈중학교,cat__next_station_효돈초등학교,cat__next_station_효돈축구공원,cat__next_station_효례교,cat__next_station_흙통,cat__next_station_희진주유소
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210452,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
210453,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
210454,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
210455,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
# 정답값(target)은 prepare_features에서 제거되므로, 원본 df_model에서 따로 가져옵니다.
y = df_model[target_col]

y

0         24
1         36
2         40
3         42
4         64
          ..
210452    96
210453    50
210454    16
210455    38
210456    24
Name: next_arrive_time, Length: 210457, dtype: int64

In [23]:
X = prepare_features(df_model)

X

,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,distance,next_station,next_latitude,next_longitude,year,month,day,dayofweek,now_hour
0,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,266.0,제대마을,33.457724,126.554014,2019,10,15,1,6.0
1,405136001,7997025,360-1,33.457724,126.554014,제대마을,333.0,제대아파트,33.458783,126.557353,2019,10,15,1,6.0
2,405136001,7997025,360-1,33.458783,126.557353,제대아파트,415.0,제주대학교,33.459893,126.561624,2019,10,15,1,6.0
3,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),578.0,제주여자중고등학교(아라방면),33.484860,126.542928,2019,10,15,1,6.0
4,405136001,7997025,360-1,33.485662,126.494923,도호동,374.0,은남동,33.485822,126.490897,2019,10,15,1,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210452,405328102,7983486,281-2,33.255783,126.577450,비석거리,528.0,삼아아파트,33.251896,126.574417,2019,10,28,0,21.0
210453,405328102,7983486,281-2,33.248595,126.568527,동문로터리,280.0,매일올레시장 7번입구,33.249753,126.565959,2019,10,28,0,21.0
210454,405328102,7983486,281-2,33.251891,126.560303,서귀포시 구 버스터미널,114.0,아랑조을거리 입구,33.251084,126.559551,2019,10,28,0,21.0
210455,405328102,7983486,281-2,33.251084,126.559551,아랑조을거리 입구,223.0,평생학습관,33.249504,126.558068,2019,10,28,0,21.0


In [24]:
# baseline이므로 시간 순서 기반 분리는 하지 않고 단순 train/test split을 사용합니다.
# X_train, y_train: 학습용 / X_test, y_test: 검증용
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

# 나뉜 학습/검증 데이터의 크기를 확인합니다.
X_train.shape, X_test.shape

((168365, 15), (42092, 15))

In [25]:
# baseline 수준의 XGBoost 회귀 모델을 정의합니다. 각 매개변수 의미를 위 표를 참고하세요.
xgb_model = XGBRegressor()

# 전처리와 모델을 하나의 파이프라인으로 묶습니다.
# 순서: preprocessor(전처리) → model(XGBoost 학습/예측)
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', xgb_model)
])

# fit(): 입력값(X_train)과 정답값(y_train)의 관계를 모델이 학습하는 단계입니다.
# 파이프라인이므로 전처리와 모델 학습이 함께(순서대로) 수행됩니다.
# 과거 운행 기록을 보며 "이런 상황이면 다음 정류장까지 대략 이만큼 걸리는구나"라는 패턴을 익힙니다.
_ = model_pipeline.fit(X_train, y_train)

print('모델 학습이 완료되었습니다.')

모델 학습이 완료되었습니다.


In [26]:
# 검증 데이터(X_test)에 대한 예측값을 구합니다. (전처리도 파이프라인이 자동으로 적용)
y_pred = model_pipeline.predict(X_test)

# 평가 지표를 계산합니다. 실제값(y_test)과 예측값(y_pred)을 비교합니다.
mae = mean_absolute_error(y_test, y_pred)

# 지표를 소수점 4자리로 보기 좋게 출력합니다.
print(f'MAE: {mae:.4f}')

MAE: 26.2774


In [27]:
# 예측할 새로운 데이터
new_data = pd.DataFrame([
    {
        'date':'2019-10-21',
        'route_id':405136001,
        'vh_id':7997025,
        'route_nm':'360-1',
        'now_latitude':33.456267,
        'now_longitude':126.551750,
        'now_station':'제주대학교입구',
        'now_arrive_time':'08시',
        'distance':300.0,
        'next_station':'제대마을',
        'next_latitude':33.457724,
        'next_longitude':126.554014
    }
])

new_data

,date,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,now_arrive_time,distance,next_station,next_latitude,next_longitude
0,2019-10-21,405136001,7997025,360-1,33.456267,126.55175,제주대학교입구,08시,300.0,제대마을,33.457724,126.554014


In [28]:
# 학습 떄와 동일한 prepare_features 함수를 예측용 데이터에도 적용합니다.
# 같은 전처리를 거쳐야 학습 떄와 feature 구조가 일치해 모델이 올바르게 예측합니다.
new_data_prepared = prepare_features(new_data)

# 학습된 파이프라인으로 예측합니다. (내부 전처리도 함꼐 자동 적용)
new_pred = model_pipeline.predict(new_data_prepared)

# 예측 결과를 보기 좋게 원본 예시 DataFrame에 새 컬럼으로 붙입니다.
result_df = new_data.copy()
result_df['predicted_next_arrive_time'] = new_pred # 모델이 예상한 다음 정류장 도착 시간
result_df['predicted_next_arrive_time'] = result_df['predicted_next_arrive_time'].round(2) # 소수점 2자리로 정리

# 핵심 컬러만 골라 예측 결과를 확입합니다.
result_df[[
    'route_nm',
    'now_station',
    'next_station',
    'now_arrive_time',
    'distance',
    'predicted_next_arrive_time'
]]

,route_nm,now_station,next_station,now_arrive_time,distance,predicted_next_arrive_time
0,360-1,제주대학교입구,제대마을,08시,300.0,37.950001
